In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# Загрузка данных
df = pd.read_csv('data.csv')

df1 = df.copy()

# Преобразование InvoiceDate в datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='%m/%d/%Y %H:%M')

# Удаление строк с возвратами (отрицательные Quantity)
df = df[df['Quantity'] > 0]

# Расчет общей стоимости каждой покупки
df['TotalSum'] = df['Quantity'] * df['UnitPrice']

# Группировка по CustomerID и InvoiceDate для получения первого месяца покупки
cohorts = df.groupby('CustomerID')['InvoiceDate'].min().reset_index()
cohorts['CohortMonth'] = cohorts['InvoiceDate'].dt.to_period('M')

# Добавление информации о когорте в основной датафрейм
df = df.merge(cohorts[['CustomerID', 'CohortMonth']], on='CustomerID')

# Создание столбца с месяцем покупки
df['OrderMonth'] = df['InvoiceDate'].dt.to_period('M')

with pd.ExcelWriter('output.xlsx') as writer:
    df1.to_excel(writer, sheet_name='Изначальные данные', index=False)
    df.to_excel(writer, sheet_name='Добавление когорт', index=False)